# AmsiNet - Step 9: five-seed context fusion and cascade (+ mixed-script lines and confusion pairs)

**Runtime:** Colab **CPU**. Needs the five seeds of Step 3a (`block|line|word|word_moe` seeds 0-4) saved on Drive. Nothing is trained.

For every seed and for both word-agent variants (plain `word`, `word_moe`) the notebook repeats the Step 6 and Step 8 procedure **separately**: fusion weights by coordinate search on that seed's validation set, tau* by the validation rule (cheapest tau keeping 90% of the gain that the line and block agents add over free context), test evaluated once. It then reports mean +/- 95% t-interval over seeds (the same protocol as the flat ViT-B baseline), plus two analyses the paper still needs: accuracy on **mixed-script versus single-script lines**, and the **remaining confusion pairs**.

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.special import softmax
from scipy import stats
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(SEEDS=[0, 1, 2, 3, 4], VARIANTS=['word', 'word_moe'], EPS=1e-4, KEEP=0.90)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat_df = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); FLAT = float(flat_df[flat_df.agent == 'Flat baseline']['GFLOPs'].iloc[0])
missing = [f'{t}_seed{s}' for s in CFG['SEEDS'] for t in ('block', 'line', 'word', 'word_moe') if not os.path.exists(f'{ROOT}/agents/{t}_seed{s}_test_probs.npy')]
print('missing agent files:', missing or 'none')

In [ ]:
# 2. Helpers (same definitions as Steps 6 and 8)
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}
def table(sp, wtag, ltag, btag):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(wtag, sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', ltag, lid), block_agent=reg('block', btag, bid),
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=100 * float(np.trace(cm) / cm.sum()), f1=100 * float(f1_from(cm).mean()))
def unc(s): p = softmax(s, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def cascade(T, tau, W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def cost(T, li, bi, GF):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float); nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())
TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]

## 3. One run = one seed and one word-agent variant

In [ ]:
def run(seed, variant, keep_preds=False):
    wtag, ltag, btag = f'{variant}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
    GF = dict(word=float(eff[variant]), line=float(eff['line']), block=float(eff['block']))
    TV, TT = table('val', wtag, ltag, btag), table('test', wtag, ltag, btag)
    def full(T, W): return T['word'] + sum(W[k] * T[k] for k in KEYS)
    vf1 = lambda W: metr(TV['y'], full(TV, W).argmax(1))['f1']
    W = {k: 0.0 for k in KEYS}
    for _ in range(2):
        for k in KEYS: W[k] = max(GRID_W, key=lambda g: vf1({**W, k: g}))
    cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1))['f1'], cost(TV, r[1], r[2], GF)))(cascade(TV, t, W)) for t in TAUS}
    f_full, f_ctx = cv[0][0], cv[1.01][0]; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
    ok = [t for t in TAUS if cv[t][0] >= target]; tau = min(ok, key=lambda t: cv[t][1]) if ok else 0.0
    out = dict(seed=seed, variant=variant, tau_star=tau, **{f'w_{k}': W[k] for k in KEYS})
    y = TT['y']; res = {}
    for name, (s, li, bi) in dict(word=(TT['word'], np.zeros(TT['nl'], bool), np.zeros(TT['nb'], bool)), context=cascade(TT, 1.01, W), cascade=cascade(TT, tau, W), full=cascade(TT, 0, W)).items():
        m = metr(y, s.argmax(1)); out[f'{name}_acc'], out[f'{name}_f1'], out[f'{name}_gflops'] = m['acc'], m['f1'], cost(TT, li, bi, GF); res[name] = s.argmax(1)
        if name == 'cascade': out['lines_pct'], out['blocks_pct'] = 100 * li.mean(), 100 * bi.mean()
    # mixed-script lines: lines whose words have more than one true script
    first = np.full(TT['nl'], -1); mixed = np.zeros(TT['nl'], bool)
    for g, yy in zip(TT['lg'], y):
        if first[g] < 0: first[g] = yy
        elif first[g] != yy: mixed[g] = True
    mw = mixed[TT['lg']]; out['mixed_line_pct'] = 100 * mixed.mean(); out['mixed_word_pct'] = 100 * mw.mean()
    for name in ('word', 'context', 'cascade', 'full'):
        out[f'{name}_acc_mixed'] = 100 * float((res[name][mw] == y[mw]).mean()); out[f'{name}_acc_single'] = 100 * float((res[name][~mw] == y[~mw]).mean())
    return (out, dict(y=y, **res, mw=mw, conf=np.exp(TT['word']).max(1), nline=np.bincount(TT['lg'])[TT['lg']], paths=META[META.split == 'test'].path.tolist())) if keep_preds else (out, None)

In [ ]:
# 4. Run all seeds and variants (about a minute each on CPU)
rows, KEEP = [], {}
for variant in CFG['VARIANTS']:
    for seed in CFG['SEEDS']:
        o, p = run(seed, variant, keep_preds=(seed == CFG['SEEDS'][0]))
        rows.append(o); print(variant, seed, 'tau*', o['tau_star'], '| word %.2f/%.2f -> cascade %.2f/%.2f (%.1f GF) | full %.2f/%.2f (%.1f GF)' % (o['word_acc'], o['word_f1'], o['cascade_acc'], o['cascade_f1'], o['cascade_gflops'], o['full_acc'], o['full_f1'], o['full_gflops']))
        if p is not None: KEEP[variant] = p
PS = pd.DataFrame(rows); PS.to_csv(f'{ROOT}/results/multiseed_per_seed.csv', index=False)

## 5. Mean +/- 95% CI over seeds (Student t)

In [ ]:
def mci(x):
    x = np.asarray(x, float); n = len(x); h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n) if n > 1 else float('nan'); return x.mean(), h
def fmt(x, d=2): m, h = mci(x); return f'{m:.{d}f} +/- {h:.{d}f}'
LAB = dict(word='word agent only', context='+ free context', cascade='cascade at tau*', full='+ line and block agents everywhere (full fusion)')
SUM = []
for variant in CFG['VARIANTS']:
    d = PS[PS.variant == variant]
    for k, lab in LAB.items(): SUM.append((variant, lab, fmt(d[f'{k}_acc']), fmt(d[f'{k}_f1']), fmt(d[f'{k}_gflops'], 1)))
SUMDF = pd.DataFrame(SUM, columns=['word agent', 'system', 'accuracy %', 'macro-F1 %', 'GFLOPs / image'])
display(SUMDF); SUMDF.to_csv(f'{ROOT}/results/multiseed_summary.csv', index=False)
print('flat ViT-B/16 (five seeds): 93.50 +/- 0.35 acc | 88.06 +/- 0.60 macro-F1 | cost %.2f GFLOPs per word' % FLAT)
for variant in CFG['VARIANTS']:
    d = PS[PS.variant == variant]
    print(variant, 'tau* per seed:', d.tau_star.tolist(), '| lines invoked %% %s | blocks invoked %% %s' % (fmt(d.lines_pct, 1), fmt(d.blocks_pct, 1)),
          '| compute saved vs full: %s %%' % fmt(100 * (1 - d.cascade_gflops / d.full_gflops), 1))
# paired seed-level difference plain - MoE (same seed index)
a, b = PS[PS.variant == 'word'].set_index('seed'), PS[PS.variant == 'word_moe'].set_index('seed')
for k in ('word_acc', 'word_f1', 'full_acc', 'full_f1', 'cascade_acc', 'cascade_f1'):
    dd = (a[k] - b[k]).values; m, h = mci(dd); print(f'plain minus MoE, {k}: {m:+.2f} +/- {h:.2f}')

## 6. Mixed-script lines and remaining confusion pairs (first seed for the confusions, all seeds for the line split)

In [ ]:
LAT = next(i for i, s in enumerate(SCRIPTS) if s.lower() in ('latin', 'english', 'roman')); print('Latin class index:', LAT, SCRIPTS[LAT], '| label order:', SCRIPTS)
for VAR in CFG['VARIANTS']:
    d = PS[PS.variant == VAR]
    print('\n=== word head:', VAR, '| share of test lines that are mixed-script: %s %% | share of test words in them: %s %%' % (fmt(d.mixed_line_pct, 1), fmt(d.mixed_word_pct, 1)))
    MX = pd.DataFrame([(LAB[k], fmt(d[f'{k}_acc_single']), fmt(d[f'{k}_acc_mixed'])) for k in LAB], columns=['system', 'accuracy % on single-script lines', 'accuracy % on mixed-script lines'])
    display(MX); MX.to_csv(f'{ROOT}/results/mixed_lines_{VAR}.csv', index=False)
    p = KEEP[VAR]; y = p['y']; rowsP = []
    for name in ('word', 'full'):
        cm = confmat(y, p[name]); np.fill_diagonal(cm, 0); tot = cm.sum()
        for i in range(C):
            for j in range(C):
                if cm[i, j]: rowsP.append((name, f'{SCRIPTS[i]} -> {SCRIPTS[j]}', int(cm[i, j]), 100 * cm[i, j] / tot))
    PR = pd.DataFrame(rowsP, columns=['system', 'confusion (true -> predicted)', 'words', '% of all errors'])
    for name in ('word', 'full'):
        print('top confusions,', name, '(total errors %d)' % (p[name] != y).sum()); display(PR[PR.system == name].sort_values('words', ascending=False).head(8).round(2))
    nlat = y != LAT; print('non-Latin words predicted Latin: word only %.2f %% -> full fusion %.2f %%' % (100 * ((p['word'] == LAT) & nlat).sum() / nlat.sum(), 100 * ((p['full'] == LAT) & nlat).sum() / nlat.sum()))
    PR.to_csv(f'{ROOT}/results/remaining_confusions_{VAR}.csv', index=False)

## 7. Why do non-Latin words become Latin? (weak-evidence check, first seed, plain word agent)

In [ ]:
VAR = 'word_moe'; p = KEEP[VAR]; y = p['y']; nonlat = y != LAT; print('word head:', VAR)
sink_w = nonlat & (p['word'] == LAT); sink_f = nonlat & (p['full'] == LAT)
print('non-Latin test words: %d | predicted Latin: word only %d (%.2f %%) -> full fusion %d (%.2f %%) | fixed by context: %d | still wrong: %d | newly wrong: %d' % (
      nonlat.sum(), sink_w.sum(), 100 * sink_w.mean() / nonlat.mean(), sink_f.sum(), 100 * sink_f.mean() / nonlat.mean(), (sink_w & ~sink_f).sum(), (sink_w & sink_f).sum(), (~sink_w & sink_f).sum()))
OTH = [i for i in range(C) if i != LAT]
per = pd.DataFrame({'script': [SCRIPTS[i] for i in OTH],
                    'words': [int((y == i).sum()) for i in OTH],
                    'to Latin, word only %': [100 * float(((y == i) & (p['word'] == LAT)).sum() / (y == i).sum()) for i in OTH],
                    'to Latin, full fusion %': [100 * float(((y == i) & (p['full'] == LAT)).sum() / (y == i).sum()) for i in OTH]}).sort_values('to Latin, word only %', ascending=False)
display(per.round(2)); per.to_csv(f'{ROOT}/results/latin_sink_per_script.csv', index=False)
# evidence strength: word-agent confidence, words in the line, crop size (read from the crop files if they exist on this runtime)
def size(path):
    try:
        from PIL import Image
        with Image.open(path) as im: return im.size
    except Exception: return (np.nan, np.nan)
rng = np.random.RandomState(0); err = np.where(sink_w)[0]; ok = np.where(nonlat & ~sink_w)[0]; ok = rng.choice(ok, min(len(ok), 2000), replace=False)
def prof(ix):
    wh = np.array([size(p['paths'][i]) for i in ix], float)
    return dict(n=len(ix), word_conf=float(np.mean(p['conf'][ix])), words_in_line=float(np.median(p['nline'][ix])), share_single_word_lines=100 * float((p['nline'][ix] == 1).mean()),
                crop_width_px=float(np.nanmedian(wh[:, 0])) if len(wh) else np.nan, crop_height_px=float(np.nanmedian(wh[:, 1])) if len(wh) else np.nan)
EV = pd.DataFrame([dict(group='non-Latin words sent to Latin by the word agent', **prof(err)), dict(group='non-Latin words classified correctly (sample of 2000)', **prof(ok))]); display(EV.round(2))
EV.to_csv(f'{ROOT}/results/latin_sink_evidence.csv', index=False)
print('If the errors have lower confidence, fewer neighbours and smaller crops, the sink is a weak-evidence effect. NaN crop sizes mean the crop files are not on this runtime (run the cell on the A100 session or after restoring the crops).')